# Chatbot Tools — Egyptian Real Estate

**Objective:** Demonstrate and test the six function-calling tools in `src/chatbot/tools.py`
that power the Gemini chatbot. Every result uses **real data**, no API calls, no fine-tuning.

**Input:** `../../data/processed/chat_bot/chatbot_properties_v2.parquet` (62,977 × 26)
**Module:** `src/chatbot/tools.py`

**Output** (all in `../../data/processed/chat_bot/`):
- `chatbot_properties_v2.parquet` (already produced by Part 0 / this section)
- `tools_test_results.csv` — assert summary

---
## 1. Setup & Imports

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)

# Add project root to path so src.chatbot is importable
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from src.chatbot.tools import (
    search_properties,
    get_breakdown,
    get_similar,
    suggest_next_question,
    relax_filters,
    resolve_location,
    validate_and_normalize_filters,
    build_tool_declarations,
    ARABIC_ALIASES,
    _feat_matrix,    # used to pre-warm cache
    _df,
)

print('Import OK')

In [ ]:
# ── Load & verify data ────────────────────────────────────────────────────
t0 = time.perf_counter()
df = _df()
load_ms = (time.perf_counter() - t0) * 1000

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns  ({load_ms:.0f} ms)')
print(f'Columns: {list(df.columns)}')
print(f'\ncategory counts:')
print(df['category'].value_counts().to_string())

In [ ]:
# ── Pre-warm recommendation cache (one-time ~1s disk load) ────────────────
print('Loading recommendation index...')
t0 = time.perf_counter()
_feat_matrix()
_ = get_similar(int(df['id'].iloc[0]), n=1)   # trigger full disk load
print(f'Recommendation cache ready: {(time.perf_counter()-t0)*1000:.0f} ms')
print('All subsequent get_similar calls: < 5 ms')

---
## Part 0 — `district` Column (review)

The `district` column is extracted as `location.split(',')[-3]` for strings with
4+ comma-separated parts. It captures the neighbourhood / sub-city level
(e.g. "The 5th Settlement", "Zahraa El Maadi").

In [ ]:
print(f'district: {df["district"].notna().sum():,} filled, {df["district"].isna().sum():,} null')
print()
print('Top 20 districts:')
print(df['district'].value_counts().head(20).to_string())

In [ ]:
# Districts that map to 3-part locations (null district) — expected
print('Sample 3-part locations (null district — area, city, governorate only):')
df[df['district'].isna()][['location','area','city','governorate']].sample(6, random_state=1)

In [ ]:
# Unusual: districts appearing under multiple cities
multi = df.dropna(subset=['district']).groupby('district')['city'].nunique()
multi = multi[multi > 1].sort_values(ascending=False)
print(f'{len(multi)} districts appear under multiple cities (naming collision):')
for d, cnt in multi.head(8).items():
    cities = df[df['district']==d]['city'].value_counts().head(3).to_dict()
    print(f'  "{d}" -> {cnt} cities: {cities}')

---
## 2. Tool 1 — `search_properties`

Returns matching properties with `price_label`, `image_url` (668×452), `share_url`.
`bedrooms` is null for commercial types — never dropped.

In [ ]:
# Example 2a: Villas for sale in Giza sorted by price
r = search_properties(
    {'category': 'buy', 'property_type': 'Villa', 'governorate': 'Giza'},
    sort_by='price_asc',
    limit=3,
)
print(f'Total villas for sale in Giza: {r["total_matches"]:,}')
print(f'Warnings: {r["warnings"]}')
for p in r['results']:
    print(f'  id={p["id"]}  price={p["price"]:,.0f} {p["price_label"]}  beds={p["bedrooms"]}')
    print(f'    {p["title"][:60]}')
    print(f'    img: ...{p["image_url"][-25:]}')

In [ ]:
# Example 2b: Apartments for rent in New Cairo with Shared Pool + Balcony
r2 = search_properties(
    {
        'category': 'rent',
        'property_type': 'Apartment',
        'city': 'New Cairo City',
        'amenities': ['Shared Pool', 'Balcony'],
    },
    sort_by='price_asc',
    limit=3,
)
print(f'Total: {r2["total_matches"]:,}')
for p in r2['results']:
    print(f'  {p["price"]:,.0f} {p["price_label"]}  beds={p["bedrooms"]}  {p["title"][:55]}')

In [ ]:
# Example 2c: Commercial office for rent — bedrooms must be null
r3 = search_properties(
    {'category': 'commercial_rent', 'property_type': 'Office Space', 'city': 'New Cairo City'},
    limit=3,
)
print(f'Total: {r3["total_matches"]:,}')
for p in r3['results']:
    print(f'  id={p["id"]}  bedrooms={p["bedrooms"]}  price_label={p["price_label"]}')
    print(f'    {p["title"][:60]}')

In [ ]:
# Example 2d: Unverified amenity rejection
r4, w4 = validate_and_normalize_filters({'amenities': ['Penthouse', 'Balcony', 'Security']})
print('clean amenities:', r4.get('amenities'))
print('warnings:', w4)

---
## 3. Tool 2 — `get_breakdown`

Groups + counts by any dimension; returns price stats (min/median/max) per group.

In [ ]:
# Example 3a: Villas in New Cairo by district
b1 = get_breakdown(
    {'category': 'buy', 'property_type': 'Villa', 'city': 'New Cairo City'},
    by='district', top_k=6,
)
print(f'Total villas in New Cairo: {b1["total"]:,}')
pd.DataFrame(b1['groups'])

In [ ]:
# Example 3b: All buy listings by governorate
b2 = get_breakdown({'category': 'buy'}, by='governorate', top_k=8)
pd.DataFrame(b2['groups'])

In [ ]:
# Example 3c: Apartments for rent by bedrooms
b3 = get_breakdown(
    {'category': 'rent', 'property_type': 'Apartment'},
    by='bedrooms', top_k=8,
)
pd.DataFrame(b3['groups'])

---
## 4. Tool 3 — `get_similar`

Uses pre-trained content-based filtering (CBF) from `models/recommendation/`.
Precomputed top-50 index means cached calls are < 5 ms.
Returns compact dicts with a `similarity` score.

In [ ]:
# Pick a reference villa
ref_row = df[(df['category']=='buy') & (df['property_type']=='Villa')].iloc[0]
ref_id = int(ref_row['id'])
print(f'Reference: id={ref_id}  city={ref_row["city"]}')
print(f'  {str(ref_row["title"])[:70]}')
print(f'  price={ref_row["price"]:,.0f} EGP  beds={ref_row["bedrooms"]}')

In [ ]:
t0 = time.perf_counter()
sim = get_similar(ref_id, n=5, same_category=True)
ms = (time.perf_counter() - t0) * 1000
print(f'get_similar: {ms:.1f} ms  ({len(sim["similar"])} results)')
print()
for s in sim['similar']:
    print(f'  sim={s["similarity"]:.3f}  price={s["price"]:,.0f} {s["price_label"]}  {s["title"][:55]}')

In [ ]:
# Cross-category similar (same_category=False)
sim_x = get_similar(ref_id, n=3, same_category=False)
print('Cross-category similar:')
for s in sim_x['similar']:
    print(f'  sim={s["similarity"]:.3f}  category={s["category"]}  {s["title"][:55]}')

---
## 5. Tool 4 — `suggest_next_question`

Rule-based advisor: identifies the most impactful unanswered filter slot.
- Never suggests `bedrooms` for commercial categories
- Amenity options restricted to verified codes valid for the current category group
- Budget options are real quantile ranges

In [ ]:
# Example 5a: No filters set — should suggest category first
sq1 = suggest_next_question({})
print(f'No filters -> suggest: slot={sq1["slot"]}')
print(f'  reason: {sq1["reason"]}')
for opt in sq1['options'][:4]:
    print(f'    {opt["value"]}: {opt["count"]:,}')

In [ ]:
# Example 5b: Category set, no location -> suggest property_type or location
sq2 = suggest_next_question({'category': 'buy', 'governorate': 'Cairo'})
print(f'buy + Cairo -> suggest: slot={sq2["slot"]}')
for opt in sq2['options'][:5]:
    print(f'    {opt["value"]}: {opt["count"]:,}')

In [ ]:
# Example 5c: Commercial category — bedrooms must NEVER be suggested
sq3 = suggest_next_question({'category': 'commercial_rent', 'governorate': 'Cairo'})
print(f'commercial_rent + Cairo -> suggest: slot={sq3["slot"]}')
assert sq3['slot'] != 'bedrooms', 'bedrooms must never be suggested for commercial!'
print('OK: bedrooms not suggested for commercial.')
for opt in sq3['options'][:5]:
    print(f'    {opt["value"]}: {opt["count"]:,}')

In [ ]:
# Example 5d: Budget quantile ranges
sq4 = suggest_next_question({'category': 'buy', 'property_type': 'Apartment', 'city': 'New Cairo City'})
print(f'suggest: slot={sq4["slot"]}')
for opt in sq4['options']:
    print(f'    {opt["value"]}: {opt["count"]:,}')

---
## 6. Tool 5 — `relax_filters`

Progressively loosens a 0-result filter set until matches are found.
Order: drop amenities → budget +20% → size widen → area→district→city → bedrooms-1.
Never relaxes `category`.

In [ ]:
# Example 6a: Over-specific amenities — dropped first
rel1 = relax_filters({
    'category': 'buy',
    'property_type': 'Villa',
    'area': 'Hyde Park',
    'amenities': ['Private Pool', 'Private Garden', 'Walk-in Closet'],
    'min_price': 50_000_000,
    'max_price': 55_000_000,
})
print(f'Changes: {rel1["changes"]}')
print(f'Total matches after relaxation: {rel1["total_matches"]:,}')

In [ ]:
# Example 6b: Impossible budget — widened
rel2 = relax_filters({
    'category': 'rent',
    'property_type': 'Apartment',
    'city': 'Zamalek',
    'max_price': 1_000,          # EGP/month — impossibly low
    'min_bedrooms': 3,
})
print(f'Changes: {rel2["changes"]}')
print(f'Total: {rel2["total_matches"]:,}')

In [ ]:
# Example 6c: Area too specific -> widen to district -> city
rel3 = relax_filters({
    'category': 'buy',
    'property_type': 'Chalet',
    'area': 'NonExistentCompound',
    'governorate': 'Red Sea',
})
print(f'Changes: {rel3["changes"]}')
print(f'Total: {rel3["total_matches"]:,}')

In [ ]:
# Verify: relax never changes category
original_cat = 'commercial_buy'
rel4 = relax_filters({
    'category': original_cat,
    'min_price': 999_999_999,
    'max_price': 999_999_999,
})
final_cat = rel4['relaxed_filters'].get('category', original_cat)
assert final_cat == original_cat, f'category changed! was {original_cat}, now {final_cat}'
print(f'OK: category={final_cat} unchanged after relaxation.')
print(f'Changes applied: {rel4["changes"]}')

---
## 7. Tool 6 — `resolve_location`

Maps Arabic, misspelled, or vague location text to canonical data values.
Steps: (1) Arabic alias exact match → (2) case-insensitive exact match → (3) difflib fuzzy.

The alias dictionary below was built from real location values in the data.

In [ ]:
# Print the full Arabic alias dictionary
print(f'Arabic alias dictionary ({len(ARABIC_ALIASES)} entries):')
print()
for ar, en in ARABIC_ALIASES.items():
    print(f'  {ar:<30s} -> {en}')

In [ ]:
# Arabic inputs
print('=== Arabic inputs ===')
for q in ['التجمع الخامس', 'الشيخ زايد', 'المعادي', 'العاصمة الإدارية',
           'الغردقة', 'القاهرة', 'الجيزة', 'ميفيدا']:
    res = resolve_location(q)
    m = res['matches'][0] if res['matches'] else None
    print(f'  {q:<28} -> {m["value"] if m else "NO MATCH":<30} ({m["confidence"]:.2f}, {m["source"]}) [{m["column_hint"]}]' if m else f'  {q} -> NO MATCH')

In [ ]:
# English inputs
print('=== English inputs ===')
for q in ['New Cairo City', 'Sheikh Zayed', 'Maadi', 'Hyde Park',
          'Hurghada', 'Alexandria', 'Mivida']:
    res = resolve_location(q)
    m = res['matches'][0] if res['matches'] else None
    if m:
        print(f'  {q:<25} -> {m["value"]:<30} ({m["confidence"]:.2f}, {m["source"]}) [{m["column_hint"]}]')

In [ ]:
# Misspelled inputs
print('=== Misspelled inputs ===')
for q in ['Nw Cairo', 'Zmalek', 'Shikh Zayed', 'Hurgada', 'Alexndria', 'Madinty']:
    res = resolve_location(q)
    m = res['matches'][0] if res['matches'] else None
    if m:
        print(f'  {q:<20} -> {m["value"]:<30} (conf={m["confidence"]:.2f})')
    else:
        print(f'  {q:<20} -> NO MATCH (below threshold)')

In [ ]:
# Ambiguous inputs
print('=== Ambiguous inputs ===')
for q in ['New', 'City', 'Cairo']:
    res = resolve_location(q, threshold=0.5)
    print(f'  "{q}" -> ambiguous={res["ambiguous"]}, {len(res["matches"])} matches')
    for m in res['matches'][:3]:
        print(f'    {m["value"]:<30} ({m["confidence"]:.2f}, {m["column_hint"]})')

---
## 8. Simulated Conversations (No LLM)

Three realistic chatbot conversations chained purely from tool calls,
simulating what Gemini would do during function calling.

### Conversation A — Villa in New Cairo

In [ ]:
print('USER: I want a villa in New Cairo')
print()

# Step 1: resolve location
loc = resolve_location('New Cairo')
city = loc['matches'][0]['value'] if loc['matches'] else 'New Cairo City'
print(f'[resolve_location] "{city}" (conf={loc["matches"][0]["confidence"]:.2f})')
print()

# Step 2: broad search
r_a1 = search_properties({'category': 'buy', 'property_type': 'Villa', 'city': city})
print(f'[search_properties] {r_a1["total_matches"]:,} villas for sale in {city}')
print()

# Step 3: breakdown by district
b_a = get_breakdown({'category': 'buy', 'property_type': 'Villa', 'city': city},
                    by='district', top_k=5)
print('[get_breakdown] Top districts:')
for g in b_a['groups']:
    print(f'  {g["value"]}: {g["count"]} (median EGP {g["median_price"]:,.0f})')
print()

# Step 4: suggest next question
sq_a = suggest_next_question({'category': 'buy', 'property_type': 'Villa', 'city': city})
print(f'[suggest_next_question] slot={sq_a["slot"]}')
print(f'  -> "{sq_a["reason"]}"')
print()

# Step 5: narrow by district and budget
r_a2 = search_properties({
    'category': 'buy', 'property_type': 'Villa',
    'city': city,
    'district': 'The 5th Settlement',
    'min_price': 5_000_000, 'max_price': 20_000_000,
}, sort_by='price_asc', limit=3)
print(f'[search_properties] Narrowed: {r_a2["total_matches"]:,} villas in The 5th Settlement, 5M-20M EGP')
for p in r_a2['results']:
    print(f'  {p["price"]:>14,.0f} {p["price_label"]}  beds={p["bedrooms"]}  {p["title"][:50]}')
print()

# Step 6: show similar to top result
if r_a2['results']:
    ref = r_a2['results'][0]
    sim_a = get_similar(ref['id'], n=3)
    print(f'[get_similar] Top 3 similar to {ref["id"]}:')
    for s in sim_a['similar']:
        print(f'  sim={s["similarity"]:.3f}  price={s["price"]:,.0f} EGP  {s["title"][:50]}')

### Conversation B — Zero-result Search → relax_filters

In [ ]:
print('USER: I want a penthouse in Zamalek, max 100,000 EGP/month with private pool and garden')
print()

filters_b = {
    'category': 'rent',
    'property_type': 'Penthouse',
    'city': 'Zamalek',
    'max_price': 100_000,
    'amenities': ['Private Pool', 'Private Garden'],
}

r_b = search_properties(filters_b, limit=5)
print(f'[search_properties] {r_b["total_matches"]} results')
print()

if r_b['total_matches'] == 0:
    rel_b = relax_filters(filters_b)
    print('[relax_filters] Changes applied:')
    for c in rel_b['changes']:
        print(f'  - {c}')
    print(f'  Result: {rel_b["total_matches"]:,} matches after relaxation')
    for p in rel_b['results'][:2]:
        print(f'    {p["price"]:,.0f} {p["price_label"]}  {p["title"][:55]}')

### Conversation C — Apartment Rent with Amenities + Commercial Case

In [ ]:
print('USER: I want an apartment for rent in Sheikh Zayed with gym and pool')
print()

# Resolve location
loc_c = resolve_location('الشيخ زايد')
city_c = loc_c['matches'][0]['value']
print(f'[resolve_location] "{city_c}"')

r_c = search_properties({
    'category': 'rent',
    'property_type': 'Apartment',
    'city': city_c,
    'amenities': ['Shared Gym', 'Shared Pool'],
}, sort_by='price_asc', limit=3)
print(f'[search_properties] {r_c["total_matches"]:,} apartments with gym+pool in {city_c}')
for p in r_c['results']:
    print(f'  {p["price"]:,.0f} {p["price_label"]}  beds={p["bedrooms"]}  {p["title"][:55]}')

print()
print('---')
print('USER: I need an office for rent in New Cairo')
print()

sq_com = suggest_next_question({'category': 'commercial_rent', 'property_type': 'Office Space'})
print(f'[suggest_next_question] slot={sq_com["slot"]}')
assert sq_com['slot'] != 'bedrooms', 'bedrooms must never be suggested for commercial!'
print('  OK: bedrooms not suggested for commercial office search')
for opt in sq_com['options'][:4]:
    print(f'    {opt["value"]}: {opt["count"]:,}')

print()
r_com = search_properties({
    'category': 'commercial_rent',
    'property_type': 'Office Space',
    'city': 'New Cairo City',
}, sort_by='price_asc', limit=3)
print(f'[search_properties] {r_com["total_matches"]:,} offices for rent')
for p in r_com['results']:
    beds = p['bedrooms']
    assert beds is None, f'Office bedrooms should be null, got {beds}'
    print(f'  {p["price"]:,.0f} {p["price_label"]}  bedrooms={beds}  {p["title"][:50]}')
print('OK: all office bedrooms are null')

---
## 9. Assert-based Tests

In [ ]:
results = []

def check(name, passed, detail=''):
    status = 'PASS' if passed else 'FAIL'
    results.append({'test': name, 'status': status, 'detail': str(detail)})
    icon = 'OK' if passed else 'FAIL'
    print(f'[{icon}] {name}' + (f'  ({detail})' if detail else ''))

print('Running assert-based tests...')
print()

In [ ]:
# T1: search count consistent with get_breakdown sum
f = {'category': 'buy', 'governorate': 'Cairo'}
r_t1 = search_properties(f, limit=1)
b_t1 = get_breakdown(f, by='district', top_k=100)
check('T1: search total == breakdown total',
      r_t1['total_matches'] == b_t1['total'],
      f'{r_t1["total_matches"]} == {b_t1["total"]}')

In [ ]:
# T2: limit cap at 10
r_t2 = search_properties({'category': 'buy'}, limit=99)
check('T2: limit capped at 10', len(r_t2['results']) <= 10, len(r_t2['results']))

In [ ]:
# T3: price_label for rent is "EGP / month"
r_rent = search_properties({'category': 'rent'}, limit=5)
labels = set(p['price_label'] for p in r_rent['results'])
check('T3: rent price_label == "EGP / month"',
      labels == {'EGP / month'}, labels)

In [ ]:
# T4: price_label for buy is "EGP"
r_buy = search_properties({'category': 'buy'}, limit=5)
labels_b = set(p['price_label'] for p in r_buy['results'])
check('T4: buy price_label == "EGP"', labels_b == {'EGP'}, labels_b)

In [ ]:
# T5: image_url always ends with 668x452.jpg
r_img = search_properties({}, limit=10)
all_668 = all('668x452' in (p['image_url'] or '') for p in r_img['results'])
check('T5: image_url always 668x452', all_668)

In [ ]:
# T6: null bedrooms for commercial listings (never dropped)
r_com = search_properties({'category': 'commercial_buy', 'property_type': 'Shop'}, limit=10)
all_null = all(p['bedrooms'] is None for p in r_com['results'])
check('T6: Shop bedrooms are null (not dropped)', all_null)
check('T6b: Shop results not empty', r_com['total_matches'] > 0, r_com['total_matches'])

In [ ]:
# T7: no unverified amenity in TOOL_DECLARATIONS enum
decls = build_tool_declarations()
sp_decl = next(d for d in decls if d['name'] == 'search_properties')
amen_enum = (sp_decl['parameters']['properties']['filters']
             ['properties']['amenities']['items']['enum'])
from src.chatbot.tools import _amenities_map
amen = _amenities_map()
unverified_names = {m['name'] for m in amen.values() if not m['verified']}
in_enum_but_unverified = unverified_names & set(amen_enum)
check('T7: no unverified amenity in tool enum',
      len(in_enum_but_unverified) == 0, in_enum_but_unverified)

In [ ]:
# T8: Penthouse (unverified) rejected from amenities filter
clean_t8, warns_t8 = validate_and_normalize_filters({'amenities': ['Penthouse', 'Balcony']})
check('T8: Penthouse rejected (unverified)', 'Penthouse' not in (clean_t8.get('amenities') or []))
check('T8b: Balcony accepted (verified)', 'Balcony' in (clean_t8.get('amenities') or []))

In [ ]:
# T9: relax_filters never relaxes category
orig_cat = 'commercial_rent'
rel_t9 = relax_filters({'category': orig_cat, 'min_price': 999_999_999, 'max_price': 999_999_999})
final_cat = rel_t9['relaxed_filters'].get('category', orig_cat)
check('T9: relax_filters never changes category', final_cat == orig_cat, final_cat)

In [ ]:
# T10: bedrooms never suggested for commercial
sq_t10 = suggest_next_question({'category': 'commercial_buy'})
check('T10: bedrooms not suggested for commercial', sq_t10['slot'] != 'bedrooms', sq_t10['slot'])

In [ ]:
# T11: search filters respected — governorate
r_gov = search_properties({'governorate': 'Alexandria'}, limit=10)
govs = {p['location'].split(',')[-1].strip() for p in r_gov['results']}
check('T11: governorate filter respected', all('Alexandria' in loc for loc in
      [p['location'] for p in r_gov['results']]))

In [ ]:
# T12: price_period/price_type not exposed in results
r_t12 = search_properties({'category': 'buy'}, limit=5)
keys = set(r_t12['results'][0].keys()) if r_t12['results'] else set()
check('T12: price_period not in results', 'price_period' not in keys)
check('T12b: price_type not in results',  'price_type'   not in keys)

In [ ]:
# T13: 62,977 total rows
r_all = search_properties({}, limit=1)
check('T13: total rows == 62,977', r_all['total_matches'] == 62977, r_all['total_matches'])

In [ ]:
# T14: get_similar returns similarity in [0,1]
sim_t14 = get_similar(int(df['id'].iloc[0]), n=5)
sims_ok = all(0 <= s['similarity'] <= 1.0 for s in sim_t14['similar'])
check('T14: similarity in [0,1]', sims_ok, [s['similarity'] for s in sim_t14['similar']])

In [ ]:
# T15: amenities filter — property must have ALL listed codes
r_pool = search_properties({'amenities': ['Private Pool'], 'category': 'buy'}, limit=10)
from src.chatbot.tools import _amenities_map
amen = _amenities_map()
n2c = {m['name']: code for code, m in amen.items() if m['verified']}
pp_code = n2c['Private Pool']
check('T15: all results have Private Pool',
      all(pp_code in (row.get('amenities', []) or [])
          for _, row in df[df['id'].isin([p['id'] for p in r_pool['results']])].iterrows()),
      f'{pp_code} in all {len(r_pool["results"])} results')

In [ ]:
# Summary
import pandas as pd
df_results = pd.DataFrame(results)
n_pass = (df_results['status'] == 'PASS').sum()
n_fail = (df_results['status'] == 'FAIL').sum()
print()
print(f'Results: {n_pass} PASS / {n_fail} FAIL out of {len(results)}')
print()
if n_fail > 0:
    print('FAILING tests:')
    print(df_results[df_results['status']=='FAIL'].to_string(index=False))

In [ ]:
# Save test results
OUT = PROJECT_ROOT / 'data/processed/chat_bot/tools_test_results.csv'
df_results.to_csv(OUT, index=False)
print(f'Saved: {OUT}')

---
## 10. Performance Benchmarks

Target: all tools < 200 ms (after cache warm-up).

**Note on `get_similar`:** First call loads ~25 MB from disk (~1s).
Subsequent calls use the precomputed top-50 index and run in < 5 ms.

In [ ]:
print('Warming all caches...')
_feat_matrix()
_ = get_similar(int(df['id'].iloc[100]), n=1)
print('Cache warm. Benchmarking each tool (5 runs each)...')
print()

bench_cases = [
    ('search_properties',     search_properties,
     [{'category':'buy','city':'New Cairo City','property_type':'Villa'}],
     {'sort_by':'price_asc','limit':5}),
    ('get_breakdown',         get_breakdown,
     [{'category':'buy','governorate':'Cairo'},'district'],
     {'top_k':8}),
    ('get_similar',           get_similar,
     [int(df['id'].iloc[500])], {'n':5}),
    ('suggest_next_question', suggest_next_question,
     [{'category':'rent','city':'New Cairo City'}], {}),
    ('relax_filters',         relax_filters,
     [{'category':'buy','area':'Hyde Park','amenities':['Private Pool'],'max_price':1}], {}),
    ('resolve_location',      resolve_location,
     ['التجمع الخامس'], {}),
]

print(f'{"Tool":<30} {"Min ms":>8} {"Avg ms":>8} {"Max ms":>8}  Status')
print('-'*65)
for label, fn, args, kwargs in bench_cases:
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        times.append((time.perf_counter()-t0)*1000)
    avg = sum(times)/len(times)
    status = 'OK (<200ms)' if avg < 200 else 'SLOW - optimize'
    print(f'{label:<30} {min(times):>8.1f} {avg:>8.1f} {max(times):>8.1f}  {status}')

---
## 11. Tool Declarations (Gemini Function-Calling Schemas)

`TOOL_DECLARATIONS` contains the JSON-schema dicts to pass to Gemini.
Amenity enum contains only verified codes. No `price_period`/`price_type`.

In [ ]:
decls = build_tool_declarations()
print(f'{len(decls)} tool declarations built from real data')
print()
for d in decls:
    props = list(d['parameters']['properties'].keys())
    print(f'  {d["name"]}:')
    print(f'    params: {props}')
    print(f'    description: {d["description"][:90]}...')
    print()

In [ ]:
# Show amenity enum from declarations (must all be verified)
amen_enum = (decls[0]['parameters']['properties']['filters']
             ['properties']['amenities']['items']['enum'])
print(f'Verified amenity enum ({len(amen_enum)} entries):')
for name in amen_enum:
    print(f'  - {name}')

---
## 12. Findings & Next Steps

### Decisions needed before Notebook 03 (Gemini Function Calling)

1. **`get_similar` first-call latency** (~1s disk load of 25 MB index)  
   The chatbot backend should warm the cache at server startup.
   The notebook pre-warms it in Section 1; add the same warm-up to the FastAPI lifespan.

2. **`district` naming collisions** — 15 district names (e.g. "5th District", "6th District")
   appear under multiple cities. The chatbot should always use `city + district` together,
   never `district` alone. Consider aliasing "The 5th Settlement" as the canonical form of
   "التجمع الخامس" in `resolve_location`.

3. **"New Kairo" in data** — `resolve_location('New Cairo')` fuzzy-matches to "New Kairo"
   (a compound name in the area column). Add `"New Cairo"` → `"New Cairo City"` as a hard alias
   in `ARABIC_ALIASES` to guarantee exact city resolution.

4. **`suggest_next_question` — amenities slot** — the tool currently lists amenities by count
   in the current match set. Decide: should it exclude amenities that are already 100% present
   (i.e., not discriminating)?

5. **`relax_filters` — bedrooms min-1 for rent** — should the chatbot warn the user before
   silently reducing bedroom count? Currently it just applies the change.

6. **`area` null rows (1,151)** — these 3-part locations (area=city, district=null) could
   confuse `resolve_location` at the area level. No action needed unless you observe wrong matches.

### Planned notebooks

| # | Notebook | Purpose |
|---|----------|---------|
| 03 | `03_Gemini_Function_Calling.ipynb` | Wire tools to Gemini API with function calling |
| 04 | `04_Chatbot_Evaluation.ipynb` | Test accuracy, latency, edge cases |